# Create Cyprus RIF Awards

Grant-pattern ingest for Cyprus Research and Innovation Foundation (RIF), `F4320330084`, from RIF's official funded-projects workbook.

- Source page: https://www.research.org.cy/en/rifs-ri-programmes/funded-projects/
- Source file: first-party XLSX `Χρηματοδοτούμενα-Έργα-2016-2025.xlsx`, linked from the RIF site.
- Script prerequisite: run `scripts/local/cyprus_rif_to_s3.py` to create `s3://openalex-ingest/awards/cyprus_rif/cyprus_rif_funded_projects.parquet`.
- Full local contractor validation on 2026-06-04: 2,459 organization-role rows collapsed to 1,535 unique RIF proposal-number awards. Coverage is 100% title/summary/coordinator/institution/amount/currency/signature-date/programme/landing-page; 93.4% start_date/start_year; 86.3% end_year.
- Amount policy: source-published `Project Requested Funding`, EUR, no imputation.
- Date policy: `start_date` is populated only when source provides a real project start date. `end_date` remains NULL; `end_year` is derived from source start date + duration months.
- Funder identity: DOI `10.13039/501100018877` is Crossref's current Cyprus RIF funder. OpenAlex currently lists this entity as country `GR`; this notebook follows the DOI-backed current RIF funder ID.
- Provenance: `cyprus_rif_funded_projects`; priority: `212`.


## Imports / Config

In [ ]:
FUNDER_ID = 4320330084
FUNDER_NAME = "Research and Innovation Foundation"
PROVENANCE = "cyprus_rif_funded_projects"
PRIORITY = 212
RAW_TABLE = "cyprus_rif_raw"
AWARDS_TABLE = "cyprus_rif_awards"
S3_PATH = "s3a://openalex-ingest/awards/cyprus_rif/cyprus_rif_funded_projects.parquet"


## Step 1: Load RIF Parquet to Staging

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cyprus_rif_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cyprus_rif/cyprus_rif_funded_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.cyprus_rif_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.cyprus_rif_raw;


In [ ]:
%sql
SELECT
    funder_award_id,
    programme,
    display_name,
    lead_name,
    executing_institution,
    amount,
    currency,
    signature_date,
    start_date,
    duration_months,
    landing_page_url
FROM openalex.awards.cyprus_rif_raw
LIMIT 10;


## Step 1.5: Raw Coverage, Key Uniqueness, Year and Money Scans

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(lead_name) AS has_lead_name,
    COUNT(lead_family_name) AS has_lead_family_name,
    COUNT(executing_institution) AS has_institution,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(TRY_CAST(signature_date AS DATE)) AS has_signature_date,
    COUNT(TRY_CAST(start_date AS DATE)) AS has_start_date,
    COUNT(TRY_CAST(TRY_CAST(start_year AS DOUBLE) AS INT)) AS has_start_year,
    COUNT(TRY_CAST(TRY_CAST(end_year AS DOUBLE) AS INT)) AS has_end_year,
    COUNT(programme) AS has_programme,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(TRY_CAST(start_date AS DATE)), COUNT(*)) * 100.0, 1) AS pct_start_date,
    MIN(TRY_CAST(TRY_CAST(start_year AS DOUBLE) AS INT)) AS min_start_year,
    MAX(TRY_CAST(TRY_CAST(start_year AS DOUBLE) AS INT)) AS max_start_year
FROM openalex.awards.cyprus_rif_raw;


In [ ]:
%sql
SELECT
    programme,
    COUNT(*) AS awards,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_eur,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_with_amount
FROM openalex.awards.cyprus_rif_raw
GROUP BY programme
ORDER BY awards DESC
LIMIT 30;


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS rows,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 2) AS min_amount,
    ROUND(percentile_approx(TRY_CAST(amount AS DOUBLE), 0.5), 2) AS median_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 2) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_amount
FROM openalex.awards.cyprus_rif_raw
GROUP BY currency;


In [ ]:
%sql
SELECT source_row_count, COUNT(*) AS awards
FROM openalex.awards.cyprus_rif_raw
GROUP BY source_row_count
ORDER BY TRY_CAST(source_row_count AS INT);


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.cyprus_rif_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id
LIMIT 20;


## Step 1.6: Funder Existence Assertion

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for funder_id=4320330084'
) AS funder_check
FROM openalex.common.funder
WHERE funder_id = 4320330084;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cyprus_rif_awards
USING delta
AS
WITH rif_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320330084
),
raw_prepped AS (
    SELECT
        *,
        NULLIF(TRIM(funder_award_id), '') AS award_id_clean,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        TRY_CAST(start_date AS DATE) AS start_date_cast,
        TRY_CAST(signature_date AS DATE) AS signature_date_cast,
        TRY_CAST(TRY_CAST(start_year AS DOUBLE) AS INT) AS start_year_int,
        TRY_CAST(TRY_CAST(end_year AS DOUBLE) AS INT) AS end_year_int,
        NULLIF(TRIM(lead_given_name), '') AS lead_given_clean,
        NULLIF(TRIM(lead_family_name), '') AS lead_family_clean,
        NULLIF(TRIM(executing_institution), '') AS lead_affiliation_name,
        NULLIF(TRIM(programme), '') AS programme_clean
    FROM openalex.awards.cyprus_rif_raw
)
SELECT
    abs(xxhash64(CONCAT('4320330084:', LOWER(TRIM(r.award_id_clean))))) % 9000000000 AS id,
    NULLIF(TRIM(r.display_name), '') AS display_name,
    NULLIF(TRIM(r.description), '') AS description,
    4320330084 AS funder_id,
    r.award_id_clean AS funder_award_id,
    CASE WHEN r.amount_double > 0 THEN r.amount_double ELSE NULL END AS amount,
    CASE WHEN r.amount_double > 0 THEN 'EUR' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    CASE
        WHEN r.programme_clean ILIKE '%DIDAKTOR%' THEN 'fellowship'
        WHEN r.programme_clean ILIKE '%Post-Doctoral%' THEN 'fellowship'
        WHEN r.programme_clean ILIKE '%PhD%' THEN 'fellowship'
        ELSE 'research'
    END AS funding_type,
    r.programme_clean AS funder_scheme,
    'cyprus_rif_funded_projects' AS provenance,
    r.start_date_cast AS start_date,
    CAST(NULL AS DATE) AS end_date,
    CASE WHEN r.start_year_int BETWEEN 1900 AND YEAR(current_date()) + 1 THEN r.start_year_int END AS start_year,
    CASE WHEN r.end_year_int BETWEEN 1900 AND YEAR(current_date()) + 10 THEN r.end_year_int END AS end_year,
    CASE
        WHEN r.lead_family_clean IS NOT NULL OR r.lead_affiliation_name IS NOT NULL THEN
            struct(
                r.lead_given_clean AS given_name,
                r.lead_family_clean AS family_name,
                CAST(NULL AS STRING) AS orcid,
                r.start_date_cast AS role_start,
                struct(
                    r.lead_affiliation_name AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    r.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320330084:', LOWER(TRIM(r.award_id_clean))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date,
    212 AS priority
FROM raw_prepped r
CROSS JOIN rif_funder f
WHERE r.award_id_clean IS NOT NULL
  AND NULLIF(TRIM(r.display_name), '') IS NOT NULL;


In [ ]:
%sql
SELECT COUNT(*) AS transformed_rows
FROM openalex.awards.cyprus_rif_awards;


## Step 3: Delete Existing Priority Rows and Insert

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cyprus_rif_funded_projects' AND priority = 212;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    212 AS priority
FROM openalex.awards.cyprus_rif_awards;


In [ ]:
%sql
SELECT COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cyprus_rif_funded_projects' AND priority = 212;


## Step 6: Verification

In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT_IF(amount > 0) AS has_positive_amount,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_affiliation,
    COUNT(start_date) AS has_start_date,
    COUNT(start_year) AS has_start_year,
    COUNT(end_year) AS has_end_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT_IF(amount > 0), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    collect_set(currency) AS currencies
FROM openalex.awards.cyprus_rif_awards;


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS awards,
    ROUND(MIN(amount), 2) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 2) AS median_amount,
    ROUND(MAX(amount), 2) AS max_amount,
    ROUND(SUM(amount), 2) AS total_amount
FROM openalex.awards.cyprus_rif_awards
GROUP BY currency;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS awards
FROM openalex.awards.cyprus_rif_awards
GROUP BY start_year
ORDER BY start_year;


In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) AS awards
FROM openalex.awards.cyprus_rif_awards
GROUP BY funder_scheme, funding_type
ORDER BY awards DESC
LIMIT 30;


In [ ]:
%sql
-- PI frequency scan: no person should dominate the dataset.
SELECT
    lead_investigator.given_name AS given,
    lead_investigator.family_name AS family,
    COUNT(*) AS awards
FROM openalex.awards.cyprus_rif_awards
GROUP BY 1, 2
ORDER BY awards DESC, family, given
LIMIT 30;


In [ ]:
%sql
SELECT
    id,
    funder_award_id,
    SUBSTRING(display_name, 1, 90) AS title,
    amount,
    currency,
    start_date,
    start_year,
    end_year,
    lead_investigator.given_name AS lead_given,
    lead_investigator.family_name AS lead_family,
    SUBSTRING(lead_investigator.affiliation.name, 1, 70) AS affiliation,
    funder_scheme
FROM openalex.awards.cyprus_rif_awards
ORDER BY start_year DESC NULLS LAST, funder_award_id
LIMIT 25;


## Handoff / Admin Notes

Contractor-complete handoff only. Contractor has no S3 or Databricks access.

Admin steps:
1. Run `scripts/local/cyprus_rif_to_s3.py` without `--skip-upload` to upload the parquet.
2. Run this notebook on Databricks and review the Step 6 verification cells.
3. After QA, mark the Cyprus RIF tracker row Complete and wire any scheduled job only after the ingest has been run and verified.
